In [1]:
import os, pickle, re
import numpy as np
import pandas as pd
import json

from models import carl, taylr
from physics.simulation import mcfm, msq
from physics.hzz import zz4l
from physics.hstar import c6

import torch
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

In [2]:
torch.set_float32_matmul_precision('high')

import matplotlib, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": "", 
})

In [ ]:
run_dir = 'run/'

In [4]:
coeffs = [1, 2, 3, 4]
n_coeffs = len(coeffs)

def get_results(run_dir, coeffs):

    """
    Load the results from the taylr runs.
    Args:
        run_dir (str): Directory where the taylr runs are stored.
        coeffs (list): List of coefficients used in the taylr runs.
    Returns:
        events_test (list): List of test events.
        scaler_x (object): Scaler for the input features.
        models (list): List of loaded TAYLR models.
        scalers_y (list): List of scalers for the output features.
    """

    # testing events and feature scaler are the same for all runs
    first_dir = os.path.join(run_dir, f'taylr_{coeffs[0]}')
    with open(os.path.join(first_dir, 'events_test.pkl'), 'rb') as f:
        events_test = pickle.load(f)
    with open(os.path.join(first_dir, 'scaler_X.pkl'), 'rb') as f:
        scaler_x = pickle.load(f)

    models = []
    scalers_y = []

    ckpt_pattern = re.compile(r'epoch=(\d+)-val_loss=.*\.ckpt')

    for c in coeffs:
        taylr_dir = os.path.join(run_dir, f'taylr_{c}')
        
        # Load scaler_y
        with open(os.path.join(taylr_dir, 'scaler_y.pkl'), 'rb') as f:
            scaler_y = pickle.load(f)
        scalers_y.append(scaler_y)

        # Find latest version directory
        logs_dir = os.path.join(taylr_dir, 'lightning_logs')
        versions = [d for d in os.listdir(logs_dir) if d.startswith('version_') and d.split('_')[-1].isdigit()]
        if not versions:
            raise FileNotFoundError(f"No version directories found in {logs_dir}")
        latest_version = max(versions, key=lambda v: int(v.split('_')[-1]))

        # Find latest checkpoint
        checkpoint_dir = os.path.join(logs_dir, latest_version, 'checkpoints')
        all_ckpts = [f for f in os.listdir(checkpoint_dir) if ckpt_pattern.match(f)]
        if not all_ckpts:
            raise FileNotFoundError(f"No matching checkpoint files found in {checkpoint_dir}")
        all_ckpts.sort(key=lambda f: int(ckpt_pattern.match(f).group(1)), reverse=True)
        latest_ckpt = os.path.join(checkpoint_dir, all_ckpts[0])

        # Load model
        model = taylr.TAYLR.load_from_checkpoint(checkpoint_path=latest_ckpt)
        models.append(model)

    return events_test, scaler_x, models, scalers_y

events_4l, scaler_X_4l, models_4l, scalers_y_4l = get_results(os.path.join(run_dir,'h4l'), coeffs)
events_2l2v, scaler_X_2l2v, models_2l2v, scalers_y_2l2v = get_results(os.path.join(run_dir,'h2l2v'), coeffs)

In [ ]:
with open('data/zz4l/xsec.json', 'r') as f:
    xs_4l = json.load(f)
with open('data/zz2l2v/xsec.json', 'r') as f:
    xs_2l2v = json.load(f)

events_4l.weights *= np.prod(xs_4l['gg_sbi']) / events_4l.weights.sum()
events_2l2v.weights *= np.prod(xs_2l2v['gg_sbi']) / events_2l2v.weights.sum() * 0.15

In [ ]:
features_4l = ['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy',
            'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy',
            'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy',
            'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']
features_2l2v = ["l1_pt", "l1_eta", "l1_phi", "l1_energy", "l2_pt", "l2_eta", "l2_phi", "l2_energy", "met", "met_phi"]

batch_size = 512

lumi = 3000.0

c6_linspace = [-20,20,201]
cHbox_linspace = [-0.02,0.05,141]

c6_space = np.linspace(*c6_linspace)
cHbox_space = np.linspace(*cHbox_linspace)

c6_val_asimov = 0.0
cHbox_val_asimov = 0.0

i_c6_asimov = np.where(c6_space==c6_val_asimov)[0][0]
i_c6_sm = np.where(c6_space==0.0)[0][0]

i_cHbox_asimov = np.where(np.round(cHbox_space,5)==cHbox_val_asimov)[0][0]
i_cHbox_sm = np.where(np.round(cHbox_space,5)==0.0)[0][0]

In [7]:
class ZeroWeightFilter():
    def __init__(self):
        pass
    def __call__(self, kinematics, components, weights, probabilities):
        return np.where(weights != 0)

events_4l = events_4l.sample(n=50_000)
events_2l2v = events_2l2v.filter(ZeroWeightFilter()).sample(n=50_000, random_state=42)

w_4l_sm = events_4l.weights.to_numpy()
w_2l2v_sm = events_2l2v.weights.to_numpy()
w_sm = np.concatenate((w_4l_sm, w_2l2v_sm))

sigma_4l_sm = np.sum(w_4l_sm)
sigma_2l2v_sm = np.sum(w_2l2v_sm)
sigma_sm = np.sum(w_sm)

In [8]:
print(np.sum(events_4l.weights == 0))
print(np.sum(events_2l2v.weights == 0))

0
0


In [9]:
c6_4l_mod = c6.Modifier(baseline=msq.Component.SBI, events=events_4l, c6_points=[-20,-10,0,10,20])
c6_2l2v_mod = c6.Modifier(baseline=msq.Component.SBI, events=events_2l2v, c6_points=[-20,-10,0,10,20])

w_4l_c6, p_4l_c6 = c6_4l_mod.modify(c6_values=c6_space)
w_2l2v_c6, p_2l2v_c6 = c6_2l2v_mod.modify(c6_values=c6_space)

coeffs_4l_true = c6_4l_mod.coefficients[:,1:]
coeffs_2l2v_true = c6_2l2v_mod.coefficients[:,1:]
coeffs_true = np.vstack((coeffs_4l_true, coeffs_2l2v_true))

In [10]:
trainer = L.Trainer(accelerator='gpu', devices=1)

dls_4l = []
for i in range(n_coeffs):
    X_taylr = scaler_X_4l.transform(events_4l.kinematics[features_4l].to_numpy())
    X_taylr = torch.tensor(X_taylr, dtype=torch.float32) 
    dl = DataLoader(TensorDataset(X_taylr), batch_size=batch_size, num_workers=8)
    dls_4l.append(dl)

coeffs_4l = []
for i in range(n_coeffs):
    coeffs_4l.append(scalers_y_4l[i].inverse_transform(torch.concatenate(trainer.predict(models_4l[i], dls_4l[i])).numpy()[:,np.newaxis]).flatten())
coeffs_4l = np.array(coeffs_4l).T

dls_2l2v = []
for i in range(n_coeffs):
    X_taylr = scaler_X_2l2v.transform(events_2l2v.kinematics[features_2l2v].to_numpy())
    X_taylr = torch.tensor(X_taylr, dtype=torch.float32) 
    dl = DataLoader(TensorDataset(X_taylr), batch_size=batch_size, num_workers=8)
    dls_2l2v.append(dl)

coeffs_2l2v = []
for i in range(n_coeffs):
    coeffs_2l2v.append(scalers_y_2l2v[i].inverse_transform(torch.concatenate(trainer.predict(models_2l2v[i], dls_2l2v[i])).numpy()[:,np.newaxis]).flatten())
coeffs_2l2v = np.array(coeffs_2l2v).T

coeffs = np.vstack((coeffs_4l, coeffs_2l2v))

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

...

Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

In [11]:
def f(c6_space, coeffs):
    coefficients = np.concatenate([np.ones((len(coeffs),1)), coeffs], axis=1)
    c6_matrix = np.vander(c6_space, coefficients.shape[1], increasing=True).T
    return np.dot(coefficients, c6_matrix)

In [12]:
print(coeffs_true.shape)
print(coeffs.shape)

(100000, 4)
(100000, 4)


In [13]:
# true estimate
w_c6 = w_sm[:,np.newaxis] * f(c6_space, coeffs_true)
w_bsm = w_c6[:,:,np.newaxis] - 2 * (w_sm[:,np.newaxis,np.newaxis] * cHbox_space[np.newaxis,np.newaxis,:])
sigma_bsm = np.sum(w_bsm, axis=0)

# NSBI estimate
w_c6_nsbi = w_sm[:,np.newaxis] * f(c6_space, coeffs)
w_bsm_nsbi = w_c6_nsbi[:,:,np.newaxis] - 2 * (w_sm[:,np.newaxis,np.newaxis] * cHbox_space[np.newaxis,np.newaxis,:])
sigma_bsm_nsbi = np.sum(w_bsm_nsbi, axis=0)  # c6 & cHbox

In [14]:
nu_sm = sigma_sm * lumi
nu_bsm = sigma_bsm * lumi
nu_bsm_nsbi = sigma_bsm_nsbi * lumi

n_asimov = sigma_bsm[i_c6_asimov,i_cHbox_asimov]*lumi

In [15]:
t_rate = -2 * nu_sm * (np.log(nu_bsm) - np.log(nu_sm)) + 2 * (nu_bsm - nu_sm) 
t_rate_nsbi = -2 * nu_sm * (np.log(nu_bsm_nsbi) - np.log(nu_sm)) + 2 * (nu_bsm_nsbi - nu_sm) 

In [16]:
p_ratio = np.divide((w_bsm / sigma_bsm), (w_sm[:,np.newaxis,np.newaxis] / sigma_sm), where = (w_sm[:,np.newaxis,np.newaxis] / sigma_sm) != 0)
p_ratio_nsbi = np.divide((w_bsm_nsbi / sigma_bsm_nsbi), (w_sm[:,np.newaxis,np.newaxis] / sigma_sm), where = (w_sm[:,np.newaxis,np.newaxis] / sigma_sm) != 0)

# p_ratio = np.where(~np.isfinite(p_ratio), p_ratio, 1)
# p_ratio_nsbi = np.where(~np.isfinite(p_ratio_nsbi), p_ratio_nsbi, 1)

In [17]:
t_shape = - 2 * np.sum(lumi * w_sm[:,np.newaxis,np.newaxis] * np.log(p_ratio) , axis=0)
t_shape_nsbi = - 2 * np.sum(lumi * w_sm[:,np.newaxis,np.newaxis] * np.log(p_ratio_nsbi) , axis=0)

/tmp/ipykernel_447274/1892482313.py:1: RuntimeWarning: invalid value encountered in log
  t_shape = - 2 * np.sum(lumi * w_sm[:,np.newaxis,np.newaxis] * np.log(p_ratio) , axis=0)


In [18]:
t = t_rate + t_shape 
t_nsbi = t_rate_nsbi + t_shape_nsbi

In [19]:
i_c6_fit = np.nanargmin(t) // cHbox_linspace[2]
i_cHbox_fit = np.nanargmin(t)  % cHbox_linspace[2]

i_c6_fit_nsbi = np.nanargmin(t_nsbi) // cHbox_linspace[2]
i_cHbox_fit_nsbi = np.nanargmin(t_nsbi)  % cHbox_linspace[2]

i_c6_fit_rate = np.nanargmin(t_rate) // cHbox_linspace[2]
i_cHbox_fit_rate = np.nanargmin(t_rate)  % cHbox_linspace[2]

t_min = t[i_c6_fit,i_cHbox_fit]
t_min_nsbi = t_nsbi[i_c6_fit_nsbi,i_cHbox_fit_nsbi]
t_min_rate = t_rate[i_c6_fit_rate,i_cHbox_fit_rate]

In [20]:
X, Y = np.meshgrid(c6_space, cHbox_space)
Z = t.T
Z_nsbi = t_nsbi.T
Z_rate = t_rate.T

In [21]:
fig = plt.figure(figsize=(6,4))

contours_rate = plt.contour(X, Y, Z_rate, levels=[t_min_rate+1,t_min_rate+4], colors='lightgrey', linestyles=['--','-'])
contours_nsbi = plt.contour(X, Y, Z_nsbi, levels=[t_min_nsbi+1,t_min_nsbi+4], colors='black', linestyles=['--','-'])
contours = plt.contour(X, Y, Z, levels=[t_min+1,t_min+4], colors='royalblue', linestyles=['--','-'])

# plt.clabel(contours, fmt=dict(zip([t_min+1,t_min+4], ['$1\sigma$', '$2\sigma$'])))

plt.scatter(c6_space[i_c6_fit], cHbox_space[i_cHbox_fit], marker='x', color='royalblue')
# plt.scatter(c6_space[i_c6_fit_nsbi], cHbox_space[i_cHbox_fit_nsbi], marker='x', color='black')

labels = [
    Line2D([0],[0],color='royalblue',label='$\\mathrm{True\\ likelihood}$'),
    Line2D([0],[0],color='black',label='$\\mathrm{NSBI\\ estimate}$'),
    Line2D([0],[0],color='lightgrey',label='$\\mathrm{Rate\\ estimate}$'),
]
plt.legend(frameon=False, handles=labels, loc='upper center', fontsize=12)

plt.xlabel('$c_{6}$', fontsize=15)
plt.ylabel('$c_{H}$', fontsize=15)

plt.xlim(-20,20)
plt.ylim(-0.01,0.02)

plt.tick_params(axis='both', labelsize=12)

plt.text(0.04 ,0.12, '$gg \\rightarrow h^{\\ast} \\rightarrow ZZ$', transform=fig.axes[0].transAxes, ha='left', va='bottom', fontsize=12)
plt.text(0.04 ,0.04, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=fig.axes[0].transAxes, ha='left', va='bottom', fontsize=12)

plt.tight_layout()
plt.savefig('nll_gg_zz.pdf', bbox_inches='tight')
plt.close()